In [1]:
import psycopg2
from sqlalchemy import create_engine, inspect
from sqlalchemy.schema import CreateTable
from decouple import config

# Configuración de la conexión a la base de datos origen
DB_USER_SRC = config("USER2_BDI_POSTGRES")
DB_PASSWORD_SRC = config("PASS2_BDI_POSTGRES")
DB_NAME_SRC = "dl_essi"
DB_PORT_SRC = "5432"
DB_HOST_SRC = config("HOST2_BDI_POSTGRES")
cadena1 = f"postgresql://{DB_USER_SRC}:{DB_PASSWORD_SRC}@{DB_HOST_SRC}:{DB_PORT_SRC}/{DB_NAME_SRC}"

# Configuración de la conexión a la base de datos destino
DB_USER_DEST = config("USER2_BDI_DW")
DB_PASSWORD_DEST = config("PASS2_BDI_DW")
DB_NAME_DEST = "dl_essi"
DB_PORT_DEST = "5432"
DB_HOST_DEST = config("HOST2_BDI_DW")
cadena2 = f"postgresql://{DB_USER_DEST}:{DB_PASSWORD_DEST}@{DB_HOST_DEST}:{DB_PORT_DEST}/{DB_NAME_DEST}"


# Conexión a la base de datos origen
conn_src = psycopg2.connect(
    dbname=DB_NAME_SRC,
    user=DB_USER_SRC,
    password=DB_PASSWORD_SRC,
    host=DB_HOST_SRC,
    port=DB_PORT_SRC
)

# Conexión a la base de datos destino
conn_dest = psycopg2.connect(
    dbname=DB_NAME_DEST,
    user=DB_USER_DEST,
    password=DB_PASSWORD_DEST,
    host=DB_HOST_DEST,
    port=DB_PORT_DEST
)

# Crear cursores
cur_src = conn_src.cursor()
cur_dest = conn_dest.cursor()

# Obtener las tablas que comienzan con 'dim_'
cur_src.execute("""
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public' ;
""")
tables = cur_src.fetchall()

# Crear las tablas en la base de datos destino
for table in tables:
    table_name = table[0]
    
    # Obtener la definición de la tabla
    cur_src.execute(f"""
    SELECT column_name, data_type, character_maximum_length, is_nullable, column_default, is_identity
    FROM information_schema.columns
    WHERE table_name = '{table_name}';
    """)
    columns = cur_src.fetchall()

    # Crear la sentencia SQL para la creación de la tabla
    create_table_sql = f"CREATE TABLE {table_name} ("
    col_defs = []
    for col in columns:
        col_def = f"{col[0]} {col[1]}"
        if col[2]:  # Si hay una longitud máxima de caracteres
            col_def += f"({col[2]})"
        if col[5] == 'YES':  # Si el campo es autogenerado
            col_def += " GENERATED ALWAYS AS IDENTITY"
        if col[3] == 'NO':
            col_def += " NOT NULL"
        if col[4]:  # Si hay un valor por defecto
            col_def += f" DEFAULT {col[4]}"
        col_defs.append(col_def)
    create_table_sql += ", ".join(col_defs) + ");"
    
    # Ejecutar la sentencia SQL en la base de datos destino
    cur_dest.execute(create_table_sql)
    conn_dest.commit()

# Cerrar cursores y conexiones
cur_src.close()
cur_dest.close()
conn_src.close()
conn_dest.close()